# 📚 Logistic Regression: Predicting Probabilities, Not Raw Numbers
> Section 13 of the Statistics & A/B Testing curriculum — final section of Part 3: Regression  
> 통계 & A/B 테스트 커리큘럼 섹션 13 — Part 3: 회귀분석의 마지막 섹션

---
# 🎯 Learning Objective (학습 목표)
Today I want to learn / 오늘 배우고 싶은 것:
- [x] Explain why plain linear regression breaks down on a binary (0/1) outcome, and how the sigmoid function fixes it<br>이진(0/1) 결과에 일반 선형회귀가 왜 무너지는지, 시그모이드 함수가 이를 어떻게 해결하는지 설명하기
- [x] Fit a logistic regression and convert a coefficient into an odds ratio<br>로지스틱회귀를 맞추고 계수를 오즈비로 변환하기
- [x] Turn predicted probabilities into class predictions with a threshold, and evaluate the result with a confusion matrix<br>예측확률을 임계값으로 클래스 예측으로 변환하고 혼동행렬로 평가하기

---
# 🧠 Concept (개념)

## What is it? (이것은 무엇인가?)
*(Explain it in your own words.)*

**EN:** Logistic regression predicts the PROBABILITY of a binary outcome (0 or 1) as a function of one or more predictors. Instead of fitting a straight line the way Sections 11-12's linear regression does, it fits an S-shaped sigmoid curve via the logit link: log(p / (1-p)) = intercept + slope × X. Solving for p guarantees the predicted probability always stays between 0 and 1 — a guarantee plain linear regression cannot make.

**KR:** 로지스틱회귀는 하나 이상의 예측변수의 함수로 이진 결과(0 또는 1)의 확률을 예측합니다. 섹션 11~12의 선형회귀처럼 직선을 맞추는 대신, 로짓 연결함수를 통해 S자 모양의 시그모이드 곡선을 맞춥니다: log(p / (1-p)) = 절편 + 기울기 × X. p를 풀면 예측확률이 항상 0과 1 사이에 머물도록 보장됩니다 -- 일반 선형회귀는 이를 보장할 수 없습니다.

### Linear vs Logistic Regression (선형회귀 vs 로지스틱회귀)

| | Linear (Sections 11-12)<br>선형(섹션 11~12) | Logistic (Section 13)<br>로지스틱(섹션 13) |
|---|---|---|
| Predicts / 예측 대상 | A continuous number<br>연속적인 숫자 | A probability in [0, 1]<br>[0, 1] 사이의 확률 |
| Link function / 연결함수 | Identity (Y = ...)<br>항등함수(Y = ...) | Logit: log(p/(1-p)) = ...<br>로짓 |
| Coefficient meaning<br>계수의 의미 | Change in Y per unit X<br>X 한 단위당 Y의 변화 | Change in log-odds per unit X<br>X 한 단위당 로그오즈의 변화 |
| Fit method / 적합 방법 | OLS (closed-form)<br>OLS(닫힌 형태) | Maximum likelihood (iterative)<br>최대우도법(반복적) |

## Why do we use it? (왜 사용하는가?)
*(When is it useful?)*

**EN:** As Example 1 will show directly, fitting plain linear regression to a 0/1 outcome produces predicted "probabilities" below 0% or above 100% for some inputs — numbers that are literally nonsensical. Logistic regression's sigmoid guarantees every prediction is a valid probability, and its coefficients convert cleanly into odds ratios, a language business stakeholders already use informally ("twice the odds").

**KR:** 예제 1이 직접 보여주듯, 0/1 결과에 일반 선형회귀를 맞추면 일부 입력값에 대해 0% 미만이거나 100%를 넘는 예측 "확률"이 나옵니다 -- 문자 그대로 말이 안 되는 숫자입니다. 로지스틱회귀의 시그모이드는 모든 예측이 유효한 확률이 되도록 보장하며, 그 계수는 비즈니스 이해관계자들이 이미 비공식적으로 사용하는 언어인 오즈비("확률이 두 배")로 깔끔하게 변환됩니다.

## When is it used in Business Analytics? (비즈니스 분석에서 언제 사용되는가?)
*(Real-world use case)*

**EN:** Predicting whether a customer will churn, whether a lead will convert, whether a transaction is fraudulent, whether an email will be opened — any outcome that's fundamentally yes/no, where the business wants both a probability (to rank and prioritize) and an interpretable driver ("each extra support ticket multiplies churn odds by X").

**KR:** 고객이 이탈할지, 리드가 전환될지, 거래가 사기인지, 이메일이 열릴지 -- 근본적으로 예/아니오인 모든 결과에서, 비즈니스는 확률(순위를 매기고 우선순위를 정하는 용도)과 해석 가능한 동인("지원 티켓이 하나 늘 때마다 이탈 오즈가 X배가 된다")을 둘 다 원합니다.

---
# 📝 Syntax (문법)

## Basic Syntax (기본 문법)

In [10]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from sklearn.metrics import confusion_matrix

np.random.seed(10)
n = 200
x = np.random.uniform(0, 10, n)
logit_true = -2 + 0.5 * x
prob_true = 1 / (1 + np.exp(-logit_true))
y = np.random.binomial(1, prob_true)

df = pd.DataFrame({'x': x, 'y': y})
model = smf.logit('y ~ x', data=df).fit(disp=0)

print(model.params)
print(f"Odds ratio per unit of x: {np.exp(model.params['x']):.3f}")
print(model.summary())

Intercept   -1.961381
x            0.565492
dtype: float64
Odds ratio per unit of x: 1.760
                           Logit Regression Results                           
Dep. Variable:                      y   No. Observations:                  200
Model:                          Logit   Df Residuals:                      198
Method:                           MLE   Df Model:                            1
Date:                Fri, 06 Nov 2026   Pseudo R-squ.:                  0.2768
Time:                        18:24:19   Log-Likelihood:                -96.394
converged:                       True   LL-Null:                       -133.29
Covariance Type:            nonrobust   LLR p-value:                 8.665e-18
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -1.9614      0.369     -5.309      0.000      -2.685      -1.237
x              0.5655      0.081      6.

## Common Variations (자주 쓰는 변형)

In [11]:
# scikit-learn's version -- same underlying model, a different interface
# scikit-learn 버전 -- 같은 모델이지만 다른 인터페이스
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression()
clf.fit(x.reshape(-1, 1), y)
print(f"sklearn coefficient: {clf.coef_[0][0]:.3f}")
print(f"predicted probability for x=5: {clf.predict_proba([[5]])[0][1]:.3f}")

sklearn coefficient: 0.562
predicted probability for x=5: 0.703


---
# 🧪 Small Examples (예제)

## Example 1 — The Probability Bounds Problem: Why Linear Regression Breaks (확률범위 문제: 선형회귀가 무너지는 이유)

In [12]:
# Business question: an e-commerce site wants to predict whether a session
# ends in a purchase (0/1) from minutes spent on the site. Someone on the
# team fits a plain linear regression "because it's simpler." What happens?
# 비즈니스 질문: 이커머스 사이트가 세션에서 구매가 발생할지(0/1)를 사이트
# 체류 시간으로 예측하려 합니다. 팀의 누군가가 "더 간단하니까" 일반
# 선형회귀를 맞춥니다. 무슨 일이 일어날까요?

np.random.seed(429)
n = 400
session_minutes = np.random.uniform(0, 30, n)
logit_true = -3 + 0.25 * session_minutes
prob_true = 1 / (1 + np.exp(-logit_true))
purchased = np.random.binomial(1, prob_true)

df = pd.DataFrame({'session_minutes': session_minutes, 'purchased': purchased})
ols_model = smf.ols('purchased ~ session_minutes', data=df).fit()
ols_preds = ols_model.predict(df)
n_out = np.sum((ols_preds < 0) | (ols_preds > 1))

print(f"Purchase rate: {purchased.mean():.1%}")
print(f"OLS coefficients: intercept={ols_model.params['Intercept']:.3f}, slope={ols_model.params['session_minutes']:.4f}")
print(f"Predicted 'probabilities' outside [0, 1]: {n_out}/{n}")
print(f"Predicted range: [{ols_preds.min():.3f}, {ols_preds.max():.3f}]")
print()
print("For the highest-minutes sessions, linear regression predicts a purchase")
print("'probability' above 100% -- a number that cannot exist. Linear regression")
print("has no mechanism to know the outcome is bounded; it just keeps extending")
print("the same straight line past 1.0 forever.")
print("체류 시간이 가장 긴 세션들에서, 선형회귀는 100%를 넘는 구매 '확률'을")
print("예측합니다 -- 존재할 수 없는 숫자입니다. 선형회귀는 결과가 범위를 갖는다는")
print("것을 알 방법이 없어서, 같은 직선을 1.0을 넘어 영원히 계속 그립니다.")

Purchase rate: 59.5%
OLS coefficients: intercept=0.057, slope=0.0361
Predicted 'probabilities' outside [0, 1]: 37/400
Predicted range: [0.058, 1.137]

For the highest-minutes sessions, linear regression predicts a purchase
'probability' above 100% -- a number that cannot exist. Linear regression
has no mechanism to know the outcome is bounded; it just keeps extending
the same straight line past 1.0 forever.
체류 시간이 가장 긴 세션들에서, 선형회귀는 100%를 넘는 구매 '확률'을
예측합니다 -- 존재할 수 없는 숫자입니다. 선형회귀는 결과가 범위를 갖는다는
것을 알 방법이 없어서, 같은 직선을 1.0을 넘어 영원히 계속 그립니다.


## Example 2 — Fitting Logistic Regression and Converting Coefficients to Odds Ratios (로지스틱회귀 적합 및 계수를 오즈비로 변환)

In [13]:
# Same data as Example 1 -- now fit the model that's actually built for a
# binary outcome, and translate its coefficient into an odds ratio.
# 예제 1과 같은 데이터 -- 이번에는 이진 결과에 실제로 맞는 모델을 적합하고,
# 계수를 오즈비로 번역.

logit_model = smf.logit('purchased ~ session_minutes', data=df).fit(disp=0)
coef = logit_model.params['session_minutes']
odds_ratio_per_min = np.exp(coef)
odds_ratio_per_10min = np.exp(coef * 10)
logit_preds = logit_model.predict(df)

print(f"Logistic coefficient (log-odds per minute): {coef:.4f}")
print(f"Odds ratio per minute: {odds_ratio_per_min:.3f}")
print(f"Odds ratio per 10 minutes: {odds_ratio_per_10min:.3f}")
print(f"Predicted probability range: [{logit_preds.min():.3f}, {logit_preds.max():.3f}]")
print()
print("Every 10 extra minutes on the site multiplies the ODDS of purchasing by")
print(f"about {odds_ratio_per_10min:.1f}x -- and unlike Example 1's linear model, every")
print("single predicted probability stays safely inside [0, 1], no matter how")
print("long the session.")
print(f"사이트에서 10분을 더 보낼 때마다 구매 오즈가 약 {odds_ratio_per_10min:.1f}배가")
print("됩니다 -- 예제 1의 선형모델과 달리, 세션이 얼마나 길든 모든 예측확률이")
print("안전하게 [0, 1] 안에 머뭅니다.")

Logistic coefficient (log-odds per minute): 0.2083
Odds ratio per minute: 1.232
Odds ratio per 10 minutes: 8.027
Predicted probability range: [0.078, 0.977]

Every 10 extra minutes on the site multiplies the ODDS of purchasing by
about 8.0x -- and unlike Example 1's linear model, every
single predicted probability stays safely inside [0, 1], no matter how
long the session.
사이트에서 10분을 더 보낼 때마다 구매 오즈가 약 8.0배가
됩니다 -- 예제 1의 선형모델과 달리, 세션이 얼마나 길든 모든 예측확률이
안전하게 [0, 1] 안에 머뭅니다.


## Example 3 — Business Odds Ratio: Support Tickets and Churn (비즈니스 오즈비: 지원 티켓과 이탈)

In [14]:
# Business question: does filing more support tickets predict a higher
# chance a customer churns? Fit a logistic regression and interpret the
# odds ratio the way a stakeholder would hear it.
# 비즈니스 질문: 지원 티켓을 더 많이 남기는 것이 고객 이탈 가능성이
# 높다는 것을 예측하는가? 로지스틱회귀를 맞추고 이해관계자가 들을 방식으로
# 오즈비를 해석.

np.random.seed(55)
n = 500
tickets = np.random.poisson(2, n)
logit_true = -1.5 + 0.55 * tickets
prob_true = 1 / (1 + np.exp(-logit_true))
churn = np.random.binomial(1, prob_true)

df_churn = pd.DataFrame({'tickets': tickets, 'churn': churn})
churn_model = smf.logit('churn ~ tickets', data=df_churn).fit(disp=0)
churn_coef = churn_model.params['tickets']
churn_or = np.exp(churn_coef)

print(f"Churn rate: {churn.mean():.1%}")
print(f"Coefficient (log-odds per ticket): {churn_coef:.4f}")
print(f"Odds ratio per additional support ticket: {churn_or:.3f}")
print()
print("Stakeholder-friendly version: 'each additional support ticket a customer")
print(f"files multiplies their odds of churning by about {churn_or:.1f}x, holding")
print("everything else fixed.' Note this is a statement about ODDS, not about")
print("probability directly -- if the baseline probability is already high, the")
print("same odds ratio moves the probability by a smaller amount.")
print("이해관계자용 표현: '고객이 지원 티켓을 하나 더 남길 때마다, 다른 모든 것이")
print(f"고정된 상태에서 이탈 오즈가 약 {churn_or:.1f}배가 된다.' 이는 확률이 아니라")
print("오즈에 대한 진술임에 주의하세요 -- 기저 확률이 이미 높다면, 같은 오즈비도")
print("확률을 더 작은 크기로 움직입니다.")

Churn rate: 40.0%
Coefficient (log-odds per ticket): 0.5451
Odds ratio per additional support ticket: 1.725

Stakeholder-friendly version: 'each additional support ticket a customer
files multiplies their odds of churning by about 1.7x, holding
everything else fixed.' Note this is a statement about ODDS, not about
probability directly -- if the baseline probability is already high, the
same odds ratio moves the probability by a smaller amount.
이해관계자용 표현: '고객이 지원 티켓을 하나 더 남길 때마다, 다른 모든 것이
고정된 상태에서 이탈 오즈가 약 1.7배가 된다.' 이는 확률이 아니라
오즈에 대한 진술임에 주의하세요 -- 기저 확률이 이미 높다면, 같은 오즈비도
확률을 더 작은 크기로 움직입니다.


## Example 4 — From Probability to Decision: Threshold and Confusion Matrix (확률에서 결정으로: 임계값과 혼동행렬)

In [15]:
# Business question: a model outputs a churn PROBABILITY for every customer,
# but the retention team needs a yes/no decision -- who gets a retention
# call? Turn probabilities into classes and check the tradeoffs.
# 비즈니스 질문: 모델은 모든 고객에 대해 이탈 확률을 출력하지만, 리텐션팀은
# 예/아니오 결정이 필요합니다 -- 누구에게 리텐션 전화를 걸까? 확률을 클래스로
# 바꾸고 트레이드오프를 확인.

from sklearn.metrics import confusion_matrix, accuracy_score

np.random.seed(77)
n = 400
tenure_months = np.random.uniform(0, 36, n)
logit_true = 2.2 - 0.15 * tenure_months
prob_true = 1 / (1 + np.exp(-logit_true))
churn_t = np.random.binomial(1, prob_true)

df_t = pd.DataFrame({'tenure_months': tenure_months, 'churn_t': churn_t})
tenure_model = smf.logit('churn_t ~ tenure_months', data=df_t).fit(disp=0)
pred_prob = tenure_model.predict(df_t)

baseline_acc = max(churn_t.mean(), 1 - churn_t.mean())

for threshold in [0.5, 0.3]:
    pred_class = (pred_prob >= threshold).astype(int)
    cm = confusion_matrix(churn_t, pred_class)
    acc = accuracy_score(churn_t, pred_class)
    tn, fp, fn, tp = cm.ravel()
    recall = tp / (tp + fn)
    print(f"Threshold {threshold}: accuracy={acc:.1%}, recall={recall:.1%}")
    print(f"  confusion matrix [[TN={tn}, FP={fp}], [FN={fn}, TP={tp}]]")

print()
print(f"Baseline accuracy (always predict the majority class): {baseline_acc:.1%}")
print("Lowering the threshold from 0.5 to 0.3 catches MORE true churners (higher")
print("recall) at the cost of flagging more non-churners too (lower accuracy) --")
print("there is no single 'correct' threshold; it depends on whether a missed")
print("churner or a wasted retention call costs the business more.")
print("임계값을 0.5에서 0.3으로 낮추면 실제 이탈자를 더 많이 잡아내지만(재현율")
print("상승) 이탈하지 않을 고객도 더 많이 플래그합니다(정확도 하락) -- '정답'")
print("임계값은 하나가 아니며, 놓친 이탈자와 낭비된 리텐션 전화 중 어느 쪽이")
print("비즈니스에 더 큰 비용인지에 따라 달라집니다.")

Threshold 0.5: accuracy=79.8%, recall=78.8%
  confusion matrix [[TN=178, FP=43], [FN=38, TP=141]]
Threshold 0.3: accuracy=76.8%, recall=89.4%
  confusion matrix [[TN=147, FP=74], [FN=19, TP=160]]

Baseline accuracy (always predict the majority class): 55.2%
Lowering the threshold from 0.5 to 0.3 catches MORE true churners (higher
recall) at the cost of flagging more non-churners too (lower accuracy) --
there is no single 'correct' threshold; it depends on whether a missed
churner or a wasted retention call costs the business more.
임계값을 0.5에서 0.3으로 낮추면 실제 이탈자를 더 많이 잡아내지만(재현율
상승) 이탈하지 않을 고객도 더 많이 플래그합니다(정확도 하락) -- '정답'
임계값은 하나가 아니며, 놓친 이탈자와 낭비된 리텐션 전화 중 어느 쪽이
비즈니스에 더 큰 비용인지에 따라 달라집니다.


## Example 5 — Practice: Apply All Four Concepts (연습: 네 가지 개념 모두 적용하기)

Fill in every `________` blank below. The cell runs with **no errors** even if you leave blanks — it just prints a reminder instead of that part's result.

아래 `________` 빈칸을 모두 채워보세요. 빈칸을 두어도 셀은 **에러 없이** 실행되며, 그 부분의 결과 대신 안내 메시지만 출력됩니다.

In [18]:
# ============================================================
# Practice -- fill in every ________ blank, then re-run the cell
# 연습 -- 아래 ________ 빈칸을 모두 채운 뒤 셀을 다시 실행하세요
# ============================================================
BLANK = "________"

def ready(*vals):
    return all(v != BLANK for v in vals)

# ---------- Part A: predict a probability from a fitted logistic model ----------
np.random.seed(90)
n_a = 350
minutes_a = np.random.uniform(0, 40, n_a)
logit_a = -4 + 0.3 * minutes_a
prob_a = 1 / (1 + np.exp(-logit_a))
signup_a = np.random.binomial(1, prob_a)
df_a = pd.DataFrame({'minutes_a': minutes_a, 'signup_a': signup_a})
model_a = smf.logit('signup_a ~ minutes_a', data=df_a).fit(disp=0)

session_minutes_a = 20   # TODO: pick a number of minutes to predict a signup
                            # probability for, e.g. 20

if ready(session_minutes_a):
    pred_prob_a = model_a.predict(pd.DataFrame({'minutes_a': [session_minutes_a]})).iloc[0]
    print(f"Predicted signup probability at {session_minutes_a} minutes: {pred_prob_a:.3f}")
else:
    print("Part A \u270f\ufe0f  Fill in session_minutes_a above, then re-run.")

# ---------- Part B: business odds ratio for a jump in support tickets ----------
np.random.seed(55)
n_b = 500
tickets_b = np.random.poisson(2, n_b)
logit_b = -1.5 + 0.55 * tickets_b
prob_b = 1 / (1 + np.exp(-logit_b))
churn_b = np.random.binomial(1, prob_b)
df_b = pd.DataFrame({'tickets_b': tickets_b, 'churn_b': churn_b})
model_b = smf.logit('churn_b ~ tickets_b', data=df_b).fit(disp=0)

ticket_increase_b = 3   # TODO: pick how many EXTRA tickets to compute the
                            # odds multiplier for, e.g. 3

if ready(ticket_increase_b):
    odds_multiplier_b = np.exp(model_b.params['tickets_b'] * ticket_increase_b)
    print(f"Odds of churn multiply by {odds_multiplier_b:.2f}x for {ticket_increase_b} extra tickets")
else:
    print("Part B \u270f\ufe0f  Fill in ticket_increase_b above, then re-run.")

# ---------- Part C: threshold tradeoff -- recall and precision at a chosen cutoff ----------

np.random.seed(77)
n_c = 400
tenure_c = np.random.uniform(0, 36, n_c)
logit_c = 2.2 - 0.15 * tenure_c
prob_c = 1 / (1 + np.exp(-logit_c))
churn_c = np.random.binomial(1, prob_c)
df_c = pd.DataFrame({'tenure_c': tenure_c, 'churn_c': churn_c})
model_c = smf.logit('churn_c ~ tenure_c', data=df_c).fit(disp=0)
pred_prob_c = model_c.predict(df_c)

threshold_c = 0.25   # TODO: pick a probability cutoff between 0 and 1, e.g. 0.25

if ready(threshold_c):
    pred_class_c = (pred_prob_c >= threshold_c).astype(int)
    cm_c = confusion_matrix(churn_c, pred_class_c)
    tn, fp, fn, tp = cm_c.ravel()
    recall_c = tp / (tp + fn)
    precision_c = tp / (tp + fp)
    print(f"At threshold {threshold_c}: recall={recall_c:.1%}, precision={precision_c:.1%}")
else:
    print("Part C \u270f\ufe0f  Fill in threshold_c above (0 to 1), then re-run.")

# ---------- Part D: match the concept to its section number ----------
concept_d = "a coefficient means 'holding other variables constant'"

section_number_d = 12   # TODO: which section number does this concept belong to? (a number)

if ready(section_number_d):
    correct_section = 12
    print(f"Concept: '{concept_d}'")
    print(f"Your answer: Section {section_number_d}")
    if section_number_d == correct_section:
        print("Correct!")
    else:
        print(f"Not quite -- this is from Section {correct_section} (Multiple Regression & Diagnostics).")
else:
    print("Part D \u270f\ufe0f  Fill in section_number_d above, then re-run.")

Predicted signup probability at 20 minutes: 0.851
Odds of churn multiply by 5.13x for 3 extra tickets
At threshold 0.25: recall=93.9%, precision=65.9%
Concept: 'a coefficient means 'holding other variables constant''
Your answer: Section 12
Correct!


---
# ⚠️ Common Mistakes (자주 하는 실수)

### Mistake 1 — Interpreting a Logistic Coefficient Like a Linear One (로지스틱 계수를 선형 계수처럼 해석)
"The coefficient is 0.55, so churn probability goes up 0.55 per ticket" confuses a LOG-ODDS coefficient with a probability. The coefficient is on the log-odds scale — it must be exponentiated into an odds ratio (Example 2/3) before it means anything to a stakeholder, and even then it describes odds, not probability directly.  
"계수가 0.55이니 티켓 하나당 이탈 확률이 0.55만큼 오른다"는 로그오즈 계수를 확률과 혼동하는 것입니다. 계수는 로그오즈 척도에 있습니다 -- 이해관계자에게 의미가 있으려면 먼저 오즈비로 지수변환해야 하며(예제 2/3), 그렇게 해도 확률이 아니라 오즈에 대한 설명입니다.

✅ **Fix / 해결:** Always exponentiate a logistic coefficient (`np.exp(coef)`) before discussing it, and label the result an odds ratio, not a probability change.

### Mistake 2 — Trusting Accuracy Alone on an Imbalanced Outcome (불균형한 결과에서 정확도만 신뢰)
As Example 4 showed, a model can look impressive on accuracy while barely beating a "just predict the majority class every time" baseline — a 79% accurate model on data that's 55% one class isn't nearly as strong as 79% sounds in isolation.  
예제 4가 보여줬듯, 모델은 정확도에서 인상적으로 보여도 "그냥 항상 다수 클래스를 예측"하는 기준선을 겨우 이기는 수준일 수 있습니다 -- 데이터가 55%가 한 클래스인 상황에서 정확도 79%는 그 숫자만 봤을 때 느껴지는 것만큼 강력하지 않습니다.

✅ **Fix / 해결:** Always compare a model's accuracy against the baseline (majority-class) accuracy, and check recall/precision alongside it, not accuracy in isolation.

### Mistake 3 — Assuming 0.5 Is Always the Right Classification Threshold (0.5가 항상 올바른 분류 임계값이라고 가정)
As Example 4's threshold comparison showed directly, moving the cutoff from 0.5 to 0.3 changed both recall and accuracy — 0.5 is a default, not a law, and the right threshold depends on whether a false negative (missed churner) or a false positive (wasted retention call) costs the business more.  
예제 4의 임계값 비교가 직접 보여줬듯, 기준을 0.5에서 0.3으로 옮기면 재현율과 정확도가 둘 다 바뀝니다 -- 0.5는 기본값일 뿐 법칙이 아니며, 올바른 임계값은 위음성(놓친 이탈자)과 위양성(낭비된 리텐션 전화) 중 어느 쪽이 비즈니스에 더 큰 비용인지에 따라 달라집니다.

✅ **Fix / 해결:** Choose the threshold based on the relative business cost of false positives vs. false negatives, not by defaulting to 0.5.

### Mistake 4 — Confusing an Odds Ratio With a Probability Ratio (오즈비를 확률비와 혼동)
An odds ratio of 1.7 does NOT mean "1.7 times more likely" in the everyday probability sense — odds and probability only converge when the baseline probability is very small; at higher baseline rates, the same odds ratio corresponds to a smaller shift in probability.  
오즈비 1.7은 일상적인 확률 의미에서 "1.7배 더 가능성이 높다"는 뜻이 아닙니다 -- 오즈와 확률은 기저 확률이 매우 작을 때만 서로 근접하며, 기저 비율이 높을수록 같은 오즈비도 더 작은 확률 변화에 대응합니다.

✅ **Fix / 해결:** When the baseline probability isn't small, report the actual predicted probabilities (Example 1/2's approach) alongside the odds ratio, not the odds ratio alone.

### Mistake 5 — Fitting Linear Regression to a Binary Outcome "Because It's Simpler" ("더 간단하다"는 이유로 이진 결과에 선형회귀 적합)
As Example 1 demonstrated with real numbers (37 out of 400 predictions landing outside [0, 1]), plain linear regression on a 0/1 outcome isn't a harmless simplification — it produces literally impossible predicted values, and its p-values and standard errors are also technically invalid for a binary outcome.  
예제 1이 실제 숫자로 보여줬듯(400건 중 37건의 예측이 [0, 1] 밖으로 벗어남), 0/1 결과에 일반 선형회귀를 쓰는 것은 무해한 단순화가 아닙니다 -- 문자 그대로 불가능한 예측값을 만들어내며, p-value와 표준오차도 이진 결과에서는 기술적으로 유효하지 않습니다.

✅ **Fix / 해결:** Use logistic regression (or another classification model) for any binary outcome, never plain linear regression, regardless of how simple the linear version looks.

### Mistake 6 — Ignoring a Convergence Warning or Perfect Separation (수렴 경고나 완전분리를 무시)
If a predictor perfectly separates the two outcome classes (e.g., every customer with 10+ tickets churned, every customer with fewer didn't), the maximum-likelihood fit can fail to converge or produce enormous, meaningless coefficients — a convergence warning is a signal to investigate, not something to silently rerun past.  
예측변수가 두 결과 클래스를 완벽하게 분리한다면(예: 티켓 10개 이상인 고객은 모두 이탈, 그보다 적은 고객은 모두 유지), 최대우도 적합이 수렴하지 못하거나 거대하고 의미 없는 계수를 만들어낼 수 있습니다 -- 수렴 경고는 조사해야 할 신호이며, 조용히 다시 실행하고 넘어갈 것이 아닙니다.

✅ **Fix / 해결:** Investigate any convergence warning immediately — check for a predictor that perfectly or near-perfectly separates the classes before trusting the fitted coefficients.

---
# 💡 Tips (팁)
Useful tips or shortcuts / 유용한 팁과 단축법

- `np.exp(coef)` is the entire recipe for turning a logistic coefficient into an odds ratio — memorize this one line, since it's the bridge between the statistical fit and any stakeholder conversation.<br>np.exp(coef)가 로지스틱 계수를 오즈비로 바꾸는 전체 방법입니다 -- 통계적 적합과 이해관계자 대화 사이의 다리이므로 이 한 줄을 외워두세요.
- `model.predict(df)` on a fitted `smf.logit` model returns PROBABILITIES, not 0/1 classes — you always choose the threshold yourself (Example 4), it's never baked into the model automatically.<br>적합된 smf.logit 모델에서 model.predict(df)는 확률을 반환하지, 0/1 클래스를 반환하지 않습니다 -- 임계값은 항상 직접 선택하는 것이며(예제 4), 모델에 자동으로 내장되어 있지 않습니다.
- `statsmodels`' `smf.logit` gives you a full `summary()` with p-values and confidence intervals (useful for explaining WHY a predictor matters); `sklearn`'s `LogisticRegression` is leaner and better suited for a production prediction pipeline — pick based on whether you need inference or deployment.<br>statsmodels의 smf.logit은 p-value와 신뢰구간이 포함된 완전한 summary()를 제공합니다(예측변수가 왜 중요한지 설명할 때 유용); sklearn의 LogisticRegression은 더 가볍고 프로덕션 예측 파이프라인에 더 적합합니다 -- 추론이 필요한지 배포가 필요한지에 따라 선택하세요.
- McFadden's pseudo R-squared (`1 - model.llf / model.llnull` in statsmodels) is the closest logistic analogue to Section 11-12's R-squared, but it's on a different scale — don't expect it to reach values as high as a typical linear R-squared.<br>McFadden 유사 R-squared(statsmodels의 1 - model.llf / model.llnull)는 섹션 11~12의 R-squared에 가장 가까운 로지스틱 대응 개념이지만 척도가 다릅니다 -- 일반적인 선형 R-squared만큼 높은 값에 도달할 것을 기대하지 마세요.
- Before reaching for logistic regression's classification output, remember an A/B test's proportions z-test (Section 8) is answering a very similar question with one categorical predictor — the two tools share the same sigmoid/odds machinery under the hood.<br>로지스틱회귀의 분류 결과를 쓰기 전에, A/B 테스트의 비율 z-검정(섹션 8)도 하나의 범주형 예측변수로 매우 비슷한 질문에 답한다는 것을 기억하세요 -- 두 도구는 내부적으로 같은 시그모이드·오즈 메커니즘을 공유합니다.

---
# 🔗 Related Concepts (관련 개념)

```
Multiple Regression & Diagnostics (Section 12)
   multiple predictors, multicollinearity/VIF, adjusted R-squared, full diagnostics
        ↓
Logistic Regression (Section 13)  ← you are here / 현재 위치
   sigmoid function, odds ratios, classification threshold, confusion matrix
        ↓
[ Part 3: Regression -- COMPLETE / Part 3: 회귀분석 -- 완료 ]
        ↓
[ Part 4: A/B Testing in Practice begins / Part 4: A/B 테스트 실무, 여기서 시작 ]
A/B Test Design -> Sample Size & Power -> Analysis -> Pitfalls
```

*How is today's topic connected to other concepts?*

**EN:** This section closes Part 3 by handling the one case Sections 11-12 couldn't: an outcome that's fundamentally binary rather than continuous. The confusion-matrix thinking in Example 4 directly foreshadows Section 8's Sample Ratio Mismatch check and Section 15's segment analysis (both are really about counts falling into categories), and the sigmoid/odds machinery built here is the same math underneath an A/B test's proportions z-test — a two-group A/B test is, structurally, logistic regression with a single 0/1 "treatment" predictor. Part 4 picks up that exact connection: designing, sizing, and analyzing experiments that are often quietly a one-predictor logistic regression in disguise.

**KR:** 이 섹션은 섹션 11~12가 다루지 못했던 한 가지 경우 -- 결과가 연속형이 아니라 근본적으로 이진형인 경우 -- 를 다루며 Part 3을 마무리합니다. 예제 4의 혼동행렬적 사고는 섹션 8의 표본 비율 불일치 확인과 섹션 15의 세그먼트 분석을 직접 예고합니다(둘 다 사실 범주에 속하는 개수에 대한 것입니다), 그리고 여기서 만든 시그모이드·오즈 메커니즘은 A/B 테스트의 비율 z-검정 밑에 있는 것과 같은 수학입니다 -- 두 그룹 A/B 테스트는 구조적으로 0/1 "처치" 예측변수 하나를 가진 로지스틱회귀입니다. Part 4는 정확히 이 연결을 이어받습니다: 종종 조용히 예측변수 하나짜리 로지스틱회귀로 변장한 실험을 설계하고, 크기를 정하고, 분석하는 것입니다.

---
# 💼 Business Example (비즈니스 예시)
*How would a Business Analyst use this?*

**Scenario / 시나리오:**

**EN:** The retention team has two signals for every subscriber: tenure in months and number of support tickets filed. They want a churn probability for every customer, a clear odds-ratio story for leadership, and a decision rule for which customers get a proactive retention call this week.

**KR:** 리텐션팀은 모든 구독자에 대해 두 가지 신호를 갖고 있습니다: 가입 기간(월)과 남긴 지원 티켓 수. 모든 고객에 대한 이탈 확률, 경영진을 위한 명확한 오즈비 설명, 그리고 이번 주에 어떤 고객에게 선제적 리텐션 전화를 걸지에 대한 결정 규칙을 원합니다.

**To-do / 할 일:**
- [x] Fit a logistic regression with tenure and ticket count as predictors<br>가입 기간과 티켓 수를 예측변수로 로지스틱회귀 맞추기
- [x] Convert each coefficient into an odds ratio leadership can understand<br>각 계수를 경영진이 이해할 수 있는 오즈비로 변환하기
- [x] Choose a classification threshold and justify it with the confusion matrix<br>분류 임계값을 선택하고 혼동행렬로 정당화하기
- [x] Recommend which customers should receive a retention call this week<br>이번 주에 어떤 고객이 리텐션 전화를 받아야 할지 제안하기

In [ ]:
from sklearn.metrics import confusion_matrix

np.random.seed(200)
n = 600
tenure_biz = np.random.uniform(0, 48, n)
tickets_biz = np.random.poisson(1.5, n)
logit_biz = 1.0 - 0.08 * tenure_biz + 0.35 * tickets_biz
prob_biz = 1 / (1 + np.exp(-logit_biz))
churn_biz = np.random.binomial(1, prob_biz)

df_biz = pd.DataFrame({'tenure_biz': tenure_biz, 'tickets_biz': tickets_biz, 'churn_biz': churn_biz})
model_biz = smf.logit('churn_biz ~ tenure_biz + tickets_biz', data=df_biz).fit(disp=0)

print("Coefficients (log-odds):")
print(model_biz.params.round(3))
print()
print("Odds ratios:")
for name, coef in model_biz.params.items():
    if name != 'Intercept':
        print(f"  {name}: {np.exp(coef):.3f}")
print()

pred_prob_biz = model_biz.predict(df_biz)
threshold_biz = 0.35
pred_class_biz = (pred_prob_biz >= threshold_biz).astype(int)
cm_biz = confusion_matrix(churn_biz, pred_class_biz)
tn, fp, fn, tp = cm_biz.ravel()
recall_biz = tp / (tp + fn)
precision_biz = tp / (tp + fp)

print(f"Churn rate: {churn_biz.mean():.1%}")
print(f"At threshold {threshold_biz}: recall={recall_biz:.1%}, precision={precision_biz:.1%}")
print(f"Confusion matrix: TN={tn}, FP={fp}, FN={fn}, TP={tp}")
print()

n_flagged = pred_class_biz.sum()
print("Recommendation: each additional support ticket multiplies churn odds by")
print(f"{np.exp(model_biz.params['tickets_biz']):.2f}x, while each extra month of tenure multiplies")
print(f"it by {np.exp(model_biz.params['tenure_biz']):.3f}x (i.e., odds fall as tenure grows). At a")
print(f"{threshold_biz} threshold, {n_flagged} of {n} customers are flagged for a retention call,")
print(f"catching {recall_biz:.0%} of true churners; lower the threshold further if a missed")
print("churner is costlier than an unnecessary call.")
print(f"제안: 지원 티켓이 하나 늘 때마다 이탈 오즈가 {np.exp(model_biz.params['tickets_biz']):.2f}배가")
print(f"되고, 가입 기간이 한 달 늘 때마다 {np.exp(model_biz.params['tenure_biz']):.3f}배가 됩니다(기간이")
print(f"길어질수록 오즈가 낮아짐). {threshold_biz} 임계값에서 {n}명 중 {n_flagged}명이 리텐션")
print(f"전화 대상으로 플래그되며, 실제 이탈자의 {recall_biz:.0%}를 잡아냅니다 -- 놓친 이탈자가")
print("불필요한 전화보다 비용이 크다면 임계값을 더 낮추세요.")

Coefficients (log-odds):
Intercept      0.977
tenure_biz    -0.085
tickets_biz    0.325
dtype: float64

Odds ratios:
  tenure_biz: 0.919
  tickets_biz: 1.385

Churn rate: 38.5%
At threshold 0.35: recall=77.1%, precision=58.7%
Confusion matrix: TN=244, FP=125, FN=53, TP=178

Recommendation: each additional support ticket multiplies churn odds by
1.38x, while each extra month of tenure multiplies
it by 0.919x (i.e., odds fall as tenure grows). At a
0.35 threshold, 303 of 600 customers are flagged for a retention call,
catching 77% of true churners; lower the threshold further if a missed
churner is costlier than an unnecessary call.
제안: 지원 티켓이 하나 늘 때마다 이탈 오즈가 1.38배가
되고, 가입 기간이 한 달 늘 때마다 0.919배가 됩니다(기간이
길어질수록 오즈가 낮아짐). 0.35 임계값에서 600명 중 303명이 리텐션
전화 대상으로 플래그되며, 실제 이탈자의 77%를 잡아냅니다 -- 놓친 이탈자가
불필요한 전화보다 비용이 크다면 임계값을 더 낮추세요.


---
# 📝 Summary (요약)
*Write today's concept in 3~5 sentences.*

**EN:** Logistic regression predicts the probability of a binary outcome by fitting a sigmoid curve through the logit link, which guarantees every prediction stays inside [0, 1] — solving the exact problem Example 1 demonstrated with plain linear regression, where 37 of 400 predicted "probabilities" landed outside that range. Its coefficients live on the log-odds scale and must be exponentiated (`np.exp(coef)`) to become odds ratios, a business-friendly number that describes how the odds of the outcome multiply per unit of a predictor — but an odds ratio is not the same as a probability ratio, and that distinction matters more as the baseline rate rises. Turning predicted probabilities into a yes/no decision requires choosing a classification threshold, and Example 4 showed this choice is a real tradeoff between recall (catching true positives) and accuracy, not a default 0.5 to apply blindly. This section closes Part 3: Regression, and its sigmoid/odds machinery reappears immediately in Part 4, since a two-group A/B test's proportions z-test is structurally the same model with a single 0/1 predictor.

**KR:** 로지스틱회귀는 로짓 연결함수를 통해 시그모이드 곡선을 맞춰 이진 결과의 확률을 예측하며, 이는 모든 예측이 [0, 1] 안에 머물도록 보장합니다 -- 예제 1이 일반 선형회귀로 보여준 정확한 문제(400건 중 37건의 예측 '확률'이 그 범위 밖으로 벗어난 것)를 해결합니다. 계수는 로그오즈 척도에 있으며 오즈비가 되려면 지수변환(np.exp(coef))이 필요합니다 -- 예측변수 한 단위당 결과의 오즈가 몇 배가 되는지 설명하는 비즈니스 친화적인 숫자이지만, 오즈비는 확률비와 같지 않으며 기저 비율이 높아질수록 이 구분이 더 중요해집니다. 예측확률을 예/아니오 결정으로 바꾸려면 분류 임계값을 선택해야 하며, 예제 4는 이 선택이 재현율(진짜 양성을 잡는 것)과 정확도 사이의 진짜 트레이드오프임을 보여줬습니다 -- 맹목적으로 적용할 기본값 0.5가 아닙니다. 이 섹션은 Part 3: 회귀분석을 마무리하며, 여기서 만든 시그모이드·오즈 메커니즘은 Part 4에서 곧바로 다시 나타납니다 -- 두 그룹 A/B 테스트의 비율 z-검정은 구조적으로 0/1 예측변수 하나를 가진 같은 모델이기 때문입니다.

---
# 📌 One Sentence Summary (한 문장 요약)
Today's topic in ONE sentence. / 오늘의 주제를 한 문장으로.

> **EN:** Logistic regression fits a sigmoid curve so every predicted probability stays inside [0, 1] — something plain linear regression can't guarantee — and converts its log-odds coefficients into odds ratios via `np.exp(coef)`, but turning those probabilities into a real yes/no decision still requires deliberately choosing a threshold based on business cost, not defaulting to 0.5.

> **KR:** 로지스틱회귀는 시그모이드 곡선을 맞춰 모든 예측확률이 [0, 1] 안에 머물도록 보장합니다 -- 일반 선형회귀는 이를 보장할 수 없습니다 -- 그리고 로그오즈 계수를 np.exp(coef)로 오즈비로 변환하지만, 그 확률을 실제 예/아니오 결정으로 바꾸려면 여전히 비즈니스 비용에 근거해 임계값을 신중하게 선택해야 하며 0.5를 기본값으로 쓸 것이 아닙니다.

---
# ❓ Review Questions (복습 질문)

**Q1 (EN).** Why does fitting plain linear regression to a 0/1 outcome sometimes produce a predicted "probability" of 1.14 or -0.05, and how does logistic regression avoid this?  
**Q1 (KR).** 0/1 결과에 일반 선형회귀를 맞추면 왜 때때로 1.14나 -0.05 같은 예측 "확률"이 나오며, 로지스틱회귀는 이를 어떻게 피하나요?

A straight line is unbounded, so it can go outside 0–1. Logistic regression models the log-odds as linear, then the sigmoid maps it back to a probability strictly between 0 and 1.  
직선은 범위 제한이 없어서 0~1을 벗어납니다. 로지스틱회귀는 로그오즈를 선형으로 모델링하고, 시그모이드로 되돌려 확률을 0과 1 사이로 만듭니다.

**Q2 (EN).** A logistic regression coefficient for "years of experience" is 0.4. Walk through the exact steps to turn that into a number you'd say out loud to a hiring manager.
**Q2 (KR).** "경력 연차"에 대한 로지스틱회귀 계수가 0.4입니다. 이를 채용 담당자에게 말할 숫자로 바꾸는 정확한 단계를 설명하세요.

Exponentiate the coefficient: exp(0.4) ≈ 1.49. Say: each extra year of experience multiplies the odds by about 1.49, about 49% higher odds (holding other variables constant).  
계수에 지수를 취합니다. exp(0.4)≈1.49입니다. "경력 1년 늘 때마다 오즈가 약 1.49배(약 49% 증가)"라고 말합니다(다른 변수 고정).

**Q3 (EN).** An odds ratio of 2.0 does NOT mean "twice as likely" in every case. When does this distinction matter most, and why?
**Q3 (KR).** 오즈비 2.0은 모든 경우에 "두 배 더 가능성이 높다"는 뜻이 아닙니다. 이 구분이 언제 가장 중요하며, 왜 그런가요?

OR 2.0 means the odds double, not the probability. The gap matters most when the base rate is high. For example, from 50% to 67% is not a doubling. For rare events (small probabilities), odds and probability are close, so "twice as likely" is roughly fine.  
오즈비 2.0은 확률이 아니라 오즈가 2배라는 뜻입니다. 기준 확률이 높을 때 차이가 가장 큽니다. 예를 들어 50%에서 67%가 되는 것은 2배가 아닙니다. 드문 사건에서는 오즈와 확률이 비슷해서 "약 2배"라고 해도 대체로 맞습니다.

**Q4 (EN).** Why might a business deliberately choose a classification threshold below 0.5, even though it lowers overall accuracy?
**Q4 (KR).** 전체 정확도가 낮아지더라도, 비즈니스가 왜 0.5보다 낮은 분류 임계값을 의도적으로 선택할 수 있나요?

When a missed positive costs far more than a false alarm (for example churn or fraud), lower the threshold to catch more positives. This raises recall and lowers precision and accuracy.  
놓친 양성(이탈, 사기 등)의 비용이 오탐 비용보다 훨씬 클 때 임계값을 낮춰 양성을 더 많이 잡습니다. 재현율은 오르고 정밀도와 정확도는 내려갑니다.

**Q5 (EN).** How is a two-group A/B test's proportions z-test related to logistic regression?
**Q5 (KR).** 두 그룹 A/B 테스트의 비율 z-검정은 로지스틱회귀와 어떻게 관련되나요?

With one binary predictor (group A or B), the logistic regression coefficient test (Wald) gives almost the same p-value as the two-proportion z-test. The coefficient is the log odds ratio between groups. Logistic regression then extends this by adding other variables.  
이진 예측변수 하나(A 또는 B 그룹)만 쓰면 로지스틱회귀의 계수 검정(Wald)은 두 비율 z-검정과 거의 같은 p-value를 줍니다. 계수는 두 그룹의 로그 오즈비입니다. 로지스틱회귀는 여기에 다른 변수를 더할 수 있어서 더 확장된 방법입니다.

---
*📅 Try answering these again in a few days. / 며칠 후 다시 답해보세요.*
*🎉 This completes Part 3: Regression. Part 4: A/B Testing in Practice starts next. / 이것으로 Part 3: 회귀분석이 완료되었습니다. 다음은 Part 4: A/B 테스트 실무입니다.*